# 第五阶段：智能指针

## 实验 6：`std::weak_ptr` —— 可检查生命周期的非拥有观察

`std::weak_ptr<T>` 观察由 `shared_ptr` 管理的对象，但不增加 strong reference count，也不决定对象何时销毁。需要访问时，通过 `lock()` 尝试取得一个临时 `shared_ptr`。

本实验关注：

- weak observer 为什么不会延长对象生命周期；
- `lock()` 如何安全区分“对象仍存活”和“对象已销毁”；
- 为什么不能用 raw pointer 替代可长期保存的 weak observer；
- 两个 strong owner 如何形成引用环；
- 如何用 `weak_ptr` 表达 parent、observer、registry 等非拥有关系；
- C++ `weak_ptr` 为什么不能直接跨 C ABI。

In [ ]:
// 本步骤：引入 shared_ptr、weak_ptr、断言、字符串和移动所需的标准库。
#include <cassert>
#include <iostream>
#include <memory>
#include <string>
#include <utility>

### 1. 准备可观察的共享对象

下面用构造、析构和存活计数确认：创建或销毁 `weak_ptr` 不影响 `User` 生命周期；只有 strong owner 的数量决定对象是否存在。

In [ ]:
// 本步骤：定义带生命周期日志的 User，观察 strong ownership 何时归零。
class User
{
public:
    User(int id, std::string name)
        : id_(id), name_(std::move(name))
    {
        // 构造开始 User 生命周期，并增加存活计数。
        ++live_count_;
        std::cout << "User " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~User() noexcept
    {
        // 最后一个 strong owner 消失时，User 才会析构。
        --live_count_;
        std::cout << "User " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    int id() const noexcept
    {
        // 返回值副本，不改变 ownership。
        return id_;
    }

    const std::string &name() const noexcept
    {
        // 返回只读借用；调用者必须先持有有效的 strong owner。
        return name_;
    }

    static int live_count() noexcept
    {
        // 返回当前存活对象数，供实验断言使用。
        return live_count_;
    }

private:
    int id_;
    std::string name_;
    inline static int live_count_ = 0;
};

### 2. weak observer 不增加 strong count

`weak_ptr` 与 `shared_ptr` 关联到同一个 control block，但它只增加 weak count，不增加 strong count：

```text
shared_ptr owner ── strong ownership ──> control block ──> User
weak_ptr observer ─ weak observation ──> control block
```

`weak_ptr` 没有 `operator->` 或 `operator*`。访问前必须先尝试建立临时 strong ownership。

In [ ]:
// 本步骤：从 shared_ptr 创建 weak observer，确认 strong count 保持不变。
{
    const int before = User::live_count();
    auto owner = std::make_shared<User>(1, "Alice");

    // observer 只关联 control block，不成为新的 strong owner。
    std::weak_ptr<User> observer = owner;
    assert(owner.use_count() == 1);
    assert(observer.use_count() == 1);
    assert(!observer.expired());
    assert(User::live_count() == before + 1);

    // 当前唯一 strong owner 仍然是 owner。
    std::cout << "strong owners = " << observer.use_count() << '\n';
}

### 3. `lock()` 在访问期间延长生命周期

`observer.lock()` 是一次完整的安全尝试：对象仍存活时返回一个新的 strong owner；对象已经销毁时返回空 `shared_ptr`。不要先调用 `expired()` 再假定下一条语句仍然有效，并发环境中状态可能在两次操作之间变化。

In [ ]:
// 本步骤：lock 一个仍存活的对象，观察临时 strong owner 的建立与释放。
{
    auto owner = std::make_shared<User>(2, "Bob");
    std::weak_ptr<User> observer = owner;
    assert(owner.use_count() == 1);

    {
        // lock 成功返回 shared_ptr，在访问期间保证 User 不会销毁。
        std::shared_ptr<User> locked = observer.lock();
        assert(locked);
        assert(locked.get() == owner.get());
        assert(owner.use_count() == 2);
        std::cout << "locked User " << locked->id() << ": " << locked->name() << '\n';
    }

    // 临时 shared_ptr 析构后，strong count 回到 1。
    assert(owner.use_count() == 1);
}

### 4. 对象销毁后 observer 变为 expired

最后一个 strong owner 消失时，`User` 立即析构。现有 `weak_ptr` 可以继续存在，因为 control block 会保留足够的信息来报告对象已经过期；直到 weak observer 也消失后，control block 才能完全释放。

In [ ]:
// 本步骤：销毁最后一个 strong owner，再通过 weak_ptr 安全检测过期状态。
{
    const int before = User::live_count();
    std::weak_ptr<User> observer;

    {
        // owner 存活期间，把同一 control block 交给 observer 观察。
        auto owner = std::make_shared<User>(3, "Carol");
        observer = owner;
        assert(!observer.expired());
        assert(User::live_count() == before + 1);
    }

    // 最后一个 strong owner 已析构，User 不再存在。
    assert(User::live_count() == before);
    assert(observer.expired());
    assert(observer.use_count() == 0);

    // lock 失败返回空 shared_ptr，不会产生悬空访问。
    std::shared_ptr<User> locked = observer.lock();
    assert(!locked);
    std::cout << "observer expired; access skipped\n";
}

raw pointer 只能保存地址。owner 析构后，它会变成悬空 pointer，既不会自动置空，也无法可靠判断对象是否还存在：

```cpp
User *observer = owner.get();
owner.reset();
observer->name(); // use-after-free：行为未定义
```

这段代码只用于阅读。raw pointer 适合同步、明确受 owner 约束的短期借用；需要跨时间保存对 shared object 的非拥有观察时，`weak_ptr` 才提供可检查的生命周期关系。

### 5. Registry 保存 weak observer

registry 或 cache 经常需要“找到对象时使用，但不能仅因登记而阻止对象销毁”。这类组件应保存 `weak_ptr`，查询时再 `lock()`。

In [ ]:
// 本步骤：定义不延长 User 生命周期的单对象 registry。
class UserRegistry
{
public:
    void observe(const std::shared_ptr<User> &user) noexcept
    {
        // 从 shared_ptr 建立 weak observation，不增加 strong count。
        user_ = user;
    }

    std::shared_ptr<User> find() const noexcept
    {
        // 查询者得到临时 strong owner；目标过期时得到空结果。
        return user_.lock();
    }

private:
    std::weak_ptr<User> user_;
};

In [ ]:
// 本步骤：验证 registry 能在对象存活时查到它，却不会阻止对象销毁。
{
    const int before = User::live_count();
    UserRegistry registry;
    auto owner = std::make_shared<User>(4, "Dora");

    // 登记 observer 不增加 strong count。
    registry.observe(owner);
    assert(owner.use_count() == 1);

    {
        // find 在本作用域内建立第二个 strong owner。
        std::shared_ptr<User> found = registry.find();
        assert(found && found->id() == 4);
        assert(owner.use_count() == 2);
    }

    // 外部 owner 结束后，registry 的 weak_ptr 不会保活 User。
    owner.reset();
    assert(User::live_count() == before);
    assert(!registry.find());
}

### 6. 两个 `shared_ptr` 会形成 strong cycle

如果对象 A strong-own B，同时 B strong-own A，即使所有外部 owner 都已释放，内部 strong count 仍不会归零：

```text
       strong             strong
Node A ───────> Node B ───────> Node A
```

下面先受控复现环，再借助预先保存的 weak observer 重新取得访问权并主动拆环，避免给 Notebook kernel 留下真实泄漏。

In [ ]:
// 本步骤：定义用 shared_ptr 双向 strong-own 对方的节点，专门复现循环引用。
class StrongNode
{
public:
    explicit StrongNode(std::string name)
        : name_(std::move(name))
    {
        // 构造日志用于确认循环存在时节点不会自动析构。
        ++live_count_;
        std::cout << "StrongNode " << name_ << " created, live = " << live_count_ << '\n';
    }

    ~StrongNode() noexcept
    {
        // 只有 strong cycle 被拆除后，析构才有机会执行。
        --live_count_;
        std::cout << "StrongNode " << name_ << " destroyed, live = " << live_count_ << '\n';
    }

    void connect(std::shared_ptr<StrongNode> peer)
    {
        // 保存 shared_ptr，使当前节点成为 peer 的 strong owner。
        peer_ = std::move(peer);
    }

    std::shared_ptr<StrongNode> peer() const
    {
        // 返回 shared_ptr 副本，让调用方临时共享 peer ownership。
        return peer_;
    }

    void disconnect() noexcept
    {
        // 主动释放内部 strong edge，用于受控拆除环。
        peer_.reset();
    }

    static int live_count() noexcept
    {
        // 返回仍存活的 StrongNode 数量。
        return live_count_;
    }

private:
    std::string name_;
    std::shared_ptr<StrongNode> peer_;
    inline static int live_count_ = 0;
};

In [ ]:
// 本步骤：建立 strong cycle，释放外部 owner 后再受控拆环。
{
    const int before = StrongNode::live_count();
    auto first = std::make_shared<StrongNode>("A");
    auto second = std::make_shared<StrongNode>("B");

    // 预先保存 weak observer；它不参与成环，只用于教学环境中重新访问节点。
    std::weak_ptr<StrongNode> observer = first;

    // 两条 strong edge 使 A 与 B 互相拥有。
    first->connect(second);
    second->connect(first);

    // 外部 owner 消失后，内部环仍使两个节点保持存活。
    first.reset();
    second.reset();
    assert(StrongNode::live_count() == before + 2);
    assert(!observer.expired());

    // lock 取得 A，再通过 A 取得 B，以便明确删除两条 strong edge。
    std::shared_ptr<StrongNode> rescued_first = observer.lock();
    std::shared_ptr<StrongNode> rescued_second = rescued_first->peer();
    rescued_first->disconnect();
    rescued_second->disconnect();

    // 清空救援 owner 后，两者都能够正常析构。
    rescued_second.reset();
    rescued_first.reset();
    assert(StrongNode::live_count() == before);
    assert(observer.expired());
}

### 7. 用 weak edge 打破循环

对象关系通常存在自然的 ownership 方向。例如 parent strong-own child，保证 child 至少与 parent 一样长寿；child 只需要知道 parent，不应反过来决定 parent 生命周期，因此保存 `weak_ptr`。

In [ ]:
// 本步骤：定义 strong child + weak parent 的非循环 ownership 图。
class TreeNode
{
public:
    explicit TreeNode(std::string name)
        : name_(std::move(name))
    {
        // 记录节点创建，便于观察整棵小树的生命周期。
        ++live_count_;
        std::cout << "TreeNode " << name_ << " created, live = " << live_count_ << '\n';
    }

    ~TreeNode() noexcept
    {
        // parent 的 strong ownership 结束后，child 也能依次析构。
        --live_count_;
        std::cout << "TreeNode " << name_ << " destroyed, live = " << live_count_ << '\n';
    }

    void set_child(std::shared_ptr<TreeNode> child)
    {
        // parent 保存 child 的 strong owner，明确向下的 ownership。
        child_ = std::move(child);
    }

    void set_parent(const std::shared_ptr<TreeNode> &parent) noexcept
    {
        // child 只观察 parent，不增加 parent 的 strong count。
        parent_ = parent;
    }

    std::shared_ptr<TreeNode> parent() const noexcept
    {
        // 访问 parent 前 lock，并把可能为空写进返回值。
        return parent_.lock();
    }

    static int live_count() noexcept
    {
        // 返回仍存活的 TreeNode 数量。
        return live_count_;
    }

private:
    std::string name_;
    std::shared_ptr<TreeNode> child_;
    std::weak_ptr<TreeNode> parent_;
    inline static int live_count_ = 0;
};

In [ ]:
// 本步骤：建立非循环 parent-child 关系，确认释放外部 owner 后自动清理。
{
    const int before = TreeNode::live_count();
    auto parent = std::make_shared<TreeNode>("parent");
    auto child = std::make_shared<TreeNode>("child");

    // parent strong-own child；child 仅 weak-observe parent。
    parent->set_child(child);
    child->set_parent(parent);
    assert(parent.use_count() == 1);
    assert(child.use_count() == 2);
    assert(child->parent().get() == parent.get());

    // 先释放 child 的外部 owner，child 仍由 parent 保活。
    child.reset();
    assert(TreeNode::live_count() == before + 2);

    // 释放 parent 时先析构 parent，再释放其 strong child 成员；没有循环阻止清理。
    parent.reset();
    assert(TreeNode::live_count() == before);
}

### 8. API、并发与 ABI 边界

C++ 内部 API 可以用类型直接表达关系：

```cpp
void retain(std::shared_ptr<User> user); // 共享并延长生命周期
void observe(std::weak_ptr<User> user);  // 保存可检查的非拥有关系
void inspect(const User &user);          // 仅在调用期间借用
```

`lock()` 对 control block 的 strong ownership 获取是并发安全的，但成功得到 `shared_ptr` 不会让 `User` 的业务字段自动线程安全。共享可变对象仍需要 mutex、atomic 或线程约束。

`weak_ptr` 和 control block 都是 C++ 标准库类型，不能直接暴露为 C ABI。C 边界通常使用 opaque handle 及明确的 `retain/release` 或查询函数。Kotlin 的 `WeakReference` 也不是 C++ control block 的直接替代品；跨语言 wrapper 必须遵循原生库定义的 handle 生命周期协议。

## 实验结论

`std::weak_ptr<T>` 应读成：**观察一个 shared object，但不延长它的生命周期；使用前通过 `lock()` 尝试取得临时 strong ownership。**

| 操作 | strong count | 语义 |
| --- | ---: | --- |
| `weak_ptr observer = owner` | 不增加 | 建立非拥有观察 |
| `observer.lock()` 成功 | 临时增加 | 在访问期间保证对象存活 |
| `observer.lock()` 失败 | 保持 0 | 对象已销毁，返回空 owner |
| `observer.expired()` | 不改变 | 只读取瞬时过期状态 |
| 销毁 `weak_ptr` | 不影响对象 | 仅结束观察关系 |

选择关系时先问：

```text
这个成员需要决定目标对象何时销毁吗？
    ├── 是：shared_ptr strong owner（必须有共享理由）
    └── 否：weak_ptr observer
```

`weak_ptr` 不是为了减少引用计数成本而存在，而是为了准确表达 non-owning relationship，并避免 shared ownership graph 中的强引用环。